In [1]:
import pandas as pd
import numpy as np
from plz_to_bundesland import get_bundesland

In [2]:
charging_stations = pd.read_csv('../data/raw/rhein-kreis-neuss-ladesaulen-in-deutschland.csv', sep=';')
charging_stations['Postleitzahl_original'] = (
    charging_stations['Postleitzahl'].copy()
)

plz = charging_stations['Postleitzahl'].astype('string').str.zfill(5)

charging_stations['Bundesland'] = plz.map(
    lambda x: get_bundesland(x) if pd.notna(x) else 'Unknown'
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].replace('Unknown', pd.NA)
)

In [3]:
postcode_fixes = [
    {'Ort': 'Weimar', 'Straße': 'Henry-van-de-Velde-Straße', 'Hausnummer': '2', 'Postleitzahl_original': 9925, 'Postleitzahl_new': '99425', 'Bundesland': 'Thüringen'},
    {'Ort': 'Roßleben-Wiehe', 'Straße': 'Am Schlifter', 'Hausnummer': '5', 'Postleitzahl_original': 6571, 'Postleitzahl_new': '06571'},
    {'Ort': 'Eisenberg', 'Straße': 'Jenaer Str.', 'Hausnummer': '36', 'Postleitzahl_original': 7607, 'Postleitzahl_new': '07607'},
    {'Ort': 'Eisenberg', 'Straße': 'Walther-Rathenau-Str.', 'Hausnummer': '3', 'Postleitzahl_original': 7607, 'Postleitzahl_new': '07607'},
    {'Ort': 'Crossen an der Elster', 'Straße': 'Lange Wiese', 'Hausnummer': '1', 'Postleitzahl_original': 7613, 'Postleitzahl_new': '07613'},
    {'Ort': 'Bad Klosterlausnitz', 'Straße': 'Jenaische Straße', 'Hausnummer': '0', 'Postleitzahl_original': 7639, 'Postleitzahl_new': '07639'},
    {'Ort': 'Paderborn', 'Straße': 'Tegelweg', 'Hausnummer': '34', 'Postleitzahl_original': 3302, 'Postleitzahl_new': '33102'},
    {'Ort': 'Köln', 'Straße': 'Aachener Straße', 'Hausnummer': '752', 'Postleitzahl_original': 5093, 'Postleitzahl_new': '50933'},
    {'Ort': 'Weilerswist', 'Straße': 'Parkallee', 'Hausnummer': '1', 'Postleitzahl_original': 5391, 'Postleitzahl_new': '53919'},
    {'Ort': 'Tübingen', 'Straße': 'Julius-Wurster-Straße', 'Hausnummer': '26', 'Postleitzahl_original': 7207, 'Postleitzahl_new': '72072'},
    {'Ort': 'Pfullingen', 'Straße': 'Marktstraße', 'Hausnummer': '57', 'Postleitzahl_original': 7279, 'Postleitzahl_new': '72793'},
    {'Ort': 'Waldlaubersheim', 'Straße': 'Auf Dem Stein', 'Hausnummer': '2', 'Postleitzahl_original': 5544, 'Postleitzahl_new': '55444'},
    {'Ort': 'Lübeck', 'Straße': 'Moislinger Allee', 'Hausnummer': '56', 'Postleitzahl_original': 2355, 'Postleitzahl_new': '23558'},
]

In [4]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_cities = (
    known.groupby('Ort')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

city_lookup = (
    known[known['Ort'].isin(valid_cities)]
    .drop_duplicates('Ort')
    .set_index('Ort')['Bundesland']
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].fillna(
        charging_stations['Ort'].map(city_lookup)
    )
)

In [5]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_districts = (
    known.groupby('Kreis/kreisfreie Stadt')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

district_lookup = (
    known[known['Kreis/kreisfreie Stadt'].isin(valid_districts)]
    .drop_duplicates('Kreis/kreisfreie Stadt')
    .set_index('Kreis/kreisfreie Stadt')['Bundesland']
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].fillna(
        charging_stations['Kreis/kreisfreie Stadt'].map(district_lookup)
    )
)

In [6]:
remaining = charging_stations[
    charging_stations['Bundesland'].isna()
]

print("Unresolved rows:", len(remaining))

display(
    remaining[
        ['Ort', 'Postleitzahl', 'Kreis/kreisfreie Stadt',
         'Straße', 'Hausnummer', 'Bundesland']
    ]
)

Unresolved rows: 1


,Ort,Postleitzahl,Kreis/kreisfreie Stadt,Straße,Hausnummer,Bundesland
59532,Weimar,9925,Kreisfreie Stadt Weimar,Henry-van-de-Velde-Straße,2,<NA>


In [7]:


charging_stations['Postleitzahl'] = charging_stations['Postleitzahl'].astype(str)
for fix in postcode_fixes:
    mask = (
        (charging_stations['Ort'] == fix['Ort'])
        & (charging_stations['Straße'] == fix['Straße'])
        & (charging_stations['Hausnummer'] == fix['Hausnummer'])
        & (charging_stations['Postleitzahl_original'] == fix['Postleitzahl_original'])
    )
    assert mask.sum() > 0, f"No station found for {fix['Ort']}, {fix['Straße']}"
    charging_stations.loc[mask, 'Postleitzahl'] = fix['Postleitzahl_new']
    if 'Bundesland' in fix:
        charging_stations.loc[mask, 'Bundesland'] = fix['Bundesland']


bundeslaender = [
    'Sachsen', 'Sachsen-Anhalt', 'Thüringen', 'Brandenburg'
]

mask = (
    (
        # charging_stations['Ort'].isin(cities) |
        charging_stations['Bundesland'].isin(bundeslaender)
    )
    & (charging_stations['Postleitzahl'].str.len() == 4)
)

bayern_mask = (
    (charging_stations['Bundesland'] == 'Bayern') &
    (charging_stations['Postleitzahl'].str.len() == 4)
)


charging_stations.loc[mask, 'Postleitzahl'] = (
    charging_stations.loc[mask, 'Postleitzahl'].str.zfill(5)
)

charging_stations.loc[bayern_mask, 'Postleitzahl'] = (
    '9' + charging_stations.loc[bayern_mask, 'Postleitzahl']
)



print(len(charging_stations[charging_stations['Postleitzahl'].astype(str).str.len() < 5]))

0


In [8]:
charging_stations['Ort'] = charging_stations['Ort'].replace(
    'Halle', 'Halle (Saale)'
)

In [9]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_cities = (
    known.groupby('Ort')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

city_lookup = (
    known[known['Ort'].isin(valid_cities)]
    .drop_duplicates('Ort')
    .set_index('Ort')['Bundesland']
)

expected = charging_stations['Ort'].map(city_lookup)

mismatches = charging_stations[
    expected.notna()
    & charging_stations['Bundesland'].notna()
    & charging_stations['Bundesland'].ne(expected).fillna(False)
]

display(
    mismatches[
        ['Ort', 'Postleitzahl', 'Bundesland',
         'Kreis/kreisfreie Stadt', 'Straße', 'Hausnummer']
    ]
)

,Ort,Postleitzahl,Bundesland,Kreis/kreisfreie Stadt,Straße,Hausnummer


In [10]:
plz = charging_stations['Postleitzahl'].astype('string')

print("Total records:", len(charging_stations))
print("Missing Bundesland:", charging_stations['Bundesland'].isna().sum())
print("Unique Bundesländer:", charging_stations['Bundesland'].nunique())
print("Missing PLZ:", plz.isna().sum())
print("Invalid PLZ format:", (~plz.str.fullmatch(r'\d{5}').fillna(False)).sum())
print("City/state mismatches:", len(mismatches))

Total records: 61793
Missing Bundesland: 0
Unique Bundesländer: 16
Missing PLZ: 0
Invalid PLZ format: 0
City/state mismatches: 0


In [11]:
connector_cols = [f'Steckertypen{i}' for i in range(1, 7)]

charging_stations_by_type = charging_stations.copy()

charging_stations_by_type['has_AC'] = (
    charging_stations_by_type[connector_cols]
    .apply(lambda col: col.str.contains('AC', na=False))
    .any(axis=1)
)

charging_stations_by_type['has_DC'] = (
    charging_stations_by_type[connector_cols]
    .apply(lambda col: col.str.contains('DC', na=False))
    .any(axis=1)
)

charging_stations_by_type


,Betreiber,Art der Ladeeinrichung,Anzahl Ladepunkte,Steckertypen1,Steckertypen2,Steckertypen3,Steckertypen4,P1 [kW],P2 [kW],P3 [kW],P4 [kW],Nennleistung Ladeeinrichtung [kW],Kreis/kreisfreie Stadt,Ort,Postleitzahl,Straße,Hausnummer,Adresszusatz,Inbetriebnahmedatum,koordinaten,Anzeigename (Karte),Steckertypen5,P5 [kW],Public Key5,Steckertypen6,P6 [kW],Public Key6,Postleitzahl_original,Bundesland,has_AC,has_DC
0,EnBW mobility+ AG und Co.KG,Normalladeeinrichtung,2.0,AC Typ 2 Steckdose; AC Schuko,AC Typ 2 Steckdose; AC Schuko,NaN,NaN,22.0,22.0,NaN,NaN,30.0,Landkreis Alb-Donau-Kreis,Langenau,89129,Karlstr.,45,NaN,NaN,"48.500374, 10.117366",NaN,NaN,NaN,NaN,NaN,NaN,NaN,89129,Baden-Württemberg,True,False
1,Joachim Stader,Normalladeeinrichtung,2.0,AC Typ 2 Fahrzeugkupplung,AC Typ 2 Fahrzeugkupplung,NaN,NaN,11.0,11.0,NaN,NaN,11.0,Landkreis Alb-Donau-Kreis,Langenau,89129,Kuftenstraße,54,NaN,NaN,"48.498206, 10.124507",NaN,NaN,NaN,NaN,NaN,NaN,NaN,89129,Baden-Württemberg,True,False
2,Johannes Stängle,Normalladeeinrichtung,2.0,AC Typ 2 Steckdose,AC Typ 2 Steckdose,NaN,NaN,11.0,11.0,NaN,NaN,11.0,Landkreis Alb-Donau-Kreis,Nerenstetten,89129,Ortsstr.,7,NaN,NaN,"48.518329, 10.119319",NaN,NaN,NaN,NaN,NaN,NaN,NaN,89129,Baden-Württemberg,True,False
3,autohaus Langenau GmbH,Normalladeeinrichtung,2.0,DC Fahrzeugkupplung Typ Combo 2 (CCS),AC Typ 2 Steckdose,NaN,NaN,22.0,11.0,NaN,NaN,33.0,Landkreis Alb-Donau-Kreis,Langenau,89129,Kiesgräble,1,NaN,NaN,"48.493378, 10.108723",NaN,NaN,NaN,NaN,NaN,NaN,NaN,89129,Baden-Württemberg,True,True
4,Citywatt GmbH,Normalladeeinrichtung,2.0,AC Typ 2 Steckdose,AC Typ 2 Steckdose,NaN,NaN,22.0,22.0,NaN,NaN,22.0,Landkreis Alb-Donau-Kreis,Machtolsheim,89150,Treffensbucher Weg,13,NaN,NaN,"48.491529, 9.750237",Citywatt GmbH,NaN,NaN,NaN,NaN,NaN,NaN,89150,Baden-Württemberg,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61788,TankE GmbH,Normalladeeinrichtung,2.0,AC Typ 2 Steckdose,AC Typ 2 Steckdose,NaN,NaN,22.0,22.0,NaN,NaN,44.0,Kreis Städteregion Aachen,Aachen,52068,Europaplatz,10,NaN,NaN,"50.779498, 6.106563",NaN,NaN,NaN,NaN,NaN,NaN,NaN,52068,Nordrhein-Westfalen,True,False
61789,Acemo GmbH,Schnellladeeinrichtung,2.0,DC Fahrzeugkupplung Typ Combo 2 (CCS),DC Fahrzeugkupplung Typ Combo 2 (CCS),NaN,NaN,50.0,50.0,NaN,NaN,50.0,Kreis Städteregion Aachen,Aachen,52070,Jülicher Strasse,318,NaN,NaN,"50.786766, 6.111505",Acemo,NaN,NaN,NaN,NaN,NaN,NaN,52070,Nordrhein-Westfalen,False,True
61790,STAWAG - Stadt- und Städteregionswerke Aachen AG,Normalladeeinrichtung,2.0,AC Typ 2 Steckdose; AC Schuko,AC Typ 2 Steckdose; AC Schuko,NaN,NaN,22.0,22.0,NaN,NaN,44.0,Kreis Städteregion Aachen,Aachen,52072,Süsterfeldstraße,200,NaN,NaN,"50.790125, 6.060361",STAWAG - Stadt- und Städteregionswerke Aachen AG,NaN,NaN,NaN,NaN,NaN,NaN,52072,Nordrhein-Westfalen,True,False
61791,STAWAG - Stadt- und Städteregionswerke Aachen AG,Normalladeeinrichtung,2.0,AC Typ 2 Steckdose,AC Typ 2 Steckdose,NaN,NaN,22.0,22.0,NaN,NaN,44.0,Kreis Städteregion Aachen,Aachen,52072,Grünenthaler Straße,55,NaN,NaN,"50.814781, 6.051971",STAWAG - Stadt- und Städteregionswerke Aachen AG,NaN,NaN,NaN,NaN,NaN,NaN,52072,Nordrhein-Westfalen,True,False


In [12]:
new_charging_stations = pd.read_excel('../data/raw/Ladesaeulenregister_BNetzA_2026-09-30.xlsx', header=10)

coords = charging_stations_by_type['koordinaten'].str.split(',', expand=True)
charging_stations_by_type['Breitengrad'] = coords[0].astype(float).round(6)
charging_stations_by_type['Längengrad'] = coords[1].astype(float).round(6)

new_charging_stations['Breitengrad'] = new_charging_stations['Breitengrad'].round(6)
new_charging_stations['Längengrad'] = new_charging_stations['Längengrad'].round(6)
new_charging_stations['Inbetriebnahmedatum'] = pd.to_datetime(
    new_charging_stations['Inbetriebnahmedatum'], dayfirst=True
)

dates_by_location = (
    new_charging_stations
    .groupby(['Breitengrad', 'Längengrad'], as_index=False)['Inbetriebnahmedatum']
    .min()
    .rename(columns={'Inbetriebnahmedatum': 'correct_date'})
)

n_before = len(charging_stations_by_type)

charging_stations_by_type = charging_stations_by_type.merge(
    dates_by_location,
    on=['Breitengrad', 'Längengrad'],
    how='left',
    validate='many_to_one'
)

assert len(charging_stations_by_type) == n_before
print(charging_stations_by_type['correct_date'].notna().mean())

charging_stations_by_type = (
    charging_stations_by_type
    .drop(columns='Inbetriebnahmedatum')
    .rename(columns={'correct_date': 'commissioning_date'})
)

0.9290372695936433


In [13]:
charging_stations_by_type.to_parquet('../data/processed/charging_stations_clean.parquet', index=False)